<a href="https://colab.research.google.com/github/SEC-API-io/sec-api-cookbook/blob/main/notebooks/fiscal-period/list-fiscal-period-end-dates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## How to Get a List of Fiscal Period End Dates of Companies

The fiscal period end date, also referred to as "period of report" in EDGAR filings, is stated in Form 10-Q, 10-K, 13F-HR and other SEC filings. In the case of SEC filings, the period of report defines the time period the filing applies to, and is reported as a single date rather than an actual time period. The property `periodOfReport` in a filing object returned by the Query API represents its value.

In the following example, the Query API is used to find and list the period of reports of companies that disclosed the 50 latest 10-K filings filed in 2023. The approach can be adapted to extract period end dates for any filing type, such as 10-Q filings, across different years.

To speed up the collection of data points, 10-K filings without their variants, such as 10-K/A and late submission notifications NT 10-K, are considered. After fetching the metadata of the filings and extracting the period end date values, a pandas DataFrame is created, holding the fiscal year end periods per CIK and per ticker of the companies, respectively.

In [ ]:
pip install sec-api

In [ ]:
from sec_api import QueryApi

queryApi = QueryApi(api_key="YOUR_API_KEY")

In [ ]:
search_params = {
  "query": "formType:\"10-K\" " +
           "AND NOT formType:\"NT 10-K\" " +
           "AND NOT formType:\"10-K/A\" " +
           "AND filedAt:[2023-01-01 TO 2023-12-31]",
  "from": "0",
  "size": "50",
  "sort": [{ "filedAt": { "order": "desc" } }]
}

response = queryApi.get_filings(search_params)

The Query API response contains a list of filings under `response["filings"]`. To inspect the response, a subset of properties from each filing—specifically `formType` and `periodOfReport`—can be printed.

For those unfamiliar with `map` and `lambda`, here's a brief explanation. The `map` function applies a `lambda` function to each filing in the list. In this case, the `lambda` function returns a new dictionary for each filing, extracting the `formType` and `periodOfReport` properties and setting them as values in the new dictionary. The result is then converted into a list for further use.

In [ ]:
list(
    map(
        lambda x: {"formType": x["formType"], "periodOfReport": x["periodOfReport"]},
        response["filings"],
    )
)[:5]

[{'formType': '10-K', 'periodOfReport': '2023-09-30'},
 {'formType': '10-K', 'periodOfReport': '2023-09-30'},
 {'formType': '10-K', 'periodOfReport': '2023-07-31'},
 {'formType': '10-K', 'periodOfReport': '2023-09-30'},
 {'formType': '10-K', 'periodOfReport': '2023-09-30'}]

The same data, containing the `formType` and `periodOfReport` properties, can also be retrieved and processed using pandas DataFrames. This approach provides a more structured and flexible way to handle and analyze the data.

In [ ]:
import pandas as pd

filings = pd.DataFrame(response["filings"])

filings[["cik", "ticker", "formType", "periodOfReport"]].head()

,cik,ticker,formType,periodOfReport
0,90168,SIF,10-K,2023-09-30
1,1967306,MSBB,10-K,2023-09-30
2,1787412,WBBA,10-K,2023-07-31
3,12040,BDL,10-K,2023-09-30
4,1929589,MRDB,10-K,2023-09-30


In [ ]:
print("Fiscal Period End Dates of Companies by CIK:")
filings[["cik", "periodOfReport"]].head()

Fiscal Period End Dates of Companies by CIK:


,cik,periodOfReport
0,90168,2023-09-30
1,1967306,2023-09-30
2,1787412,2023-07-31
3,12040,2023-09-30
4,1929589,2023-09-30


In [ ]:
print("Fiscal Period End Dates of Companies by Ticker:")
filings[["ticker", "periodOfReport"]].head()

Fiscal Period End Dates of Companies by Ticker:


,ticker,periodOfReport
0,SIF,2023-09-30
1,MSBB,2023-09-30
2,WBBA,2023-07-31
3,BDL,2023-09-30
4,MRDB,2023-09-30
